# CardioAlert — Entrenamiento corregido del modelo CNN-BiLSTM

**Autores:** Pérez · Sara — UPC, Ingeniería de Sistemas

Este notebook reemplaza al entrenamiento anterior (`TP2_CNN_Perez_Sara.ipynb`) y corrige el problema que hacía que el modelo reconociera **la base de datos de origen** en lugar de la patología:

| Problema del notebook anterior | Corrección en este notebook |
|---|---|
| Cada clase venía de una base distinta (Normal=MIT-BIH, AFib=VitalDB, Isquemia=European ST-T) | **Cada base aporta casos normales y patológicos**: la base deja de delatar la clase |
| Frecuencias distintas por clase (360 / 500 / 250 Hz) sin remuestrear | Todo se remuestrea a **250 Hz, ventanas de 10 s (2500 muestras)** |
| Preprocesamiento distinto al de la app | **Mismo código que la app**, verificado número por número (diferencia < 1e-7) |
| División aleatoria por segmento, con ventanas solapadas | Ventanas **sin solapamiento** y división **por paciente** (ningún paciente de prueba se ve al entrenar) |
| Solo CNN | **CNN-BiLSTM**, como describe la tesis |

**Datos** (públicos, se descargan de PhysioNet, no hace falta Drive):

| Base | Aporta | Frecuencia |
|---|---|---|
| MIT-BIH Arrhythmia (`mitdb`) | Normal + AFib | 360 Hz |
| MIT-BIH Atrial Fibrillation (`afdb`) | Normal + AFib | 250 Hz |
| European ST-T (`edb`) | Normal + Isquemia | 250 Hz |

**Cómo usarlo:** *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*, y luego *Entorno de ejecución → Ejecutar todo*. Al inicio pide permiso para usar tu Drive: ahí se guardan los datos descargados, así que si Colab se desconecta, al volver a ejecutar no se descarga nada de nuevo.

**Duración:** la descarga (~1.2 GB desde PhysioNet) es lo más lento, entre 30 y 90 min según la velocidad de PhysioNet. El entrenamiento con GPU toma ~10–20 min.

**Resultado:** el archivo `cardioalert_cnn_bilstm.keras`, que reemplaza al modelo de la API en Railway. Es esperable una precisión entre 85% y 95%: una cifra honesta, medida con pacientes que el modelo nunca vio.

## 1. Entorno

In [ ]:
!pip install -q wfdb

import os, json, random, time
import numpy as np
import tensorflow as tf
import wfdb

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NO — activa T4 GPU en Entorno de ejecución")
# La API en Railway usa TensorFlow 2.20. Si aquí sale otra versión mayor, avisar para alinearlas.

## 2. Configuración

`QUICK_TEST = True` usa solo 3 registros y 1 época: sirve para comprobar que todo corre en minutos. Para el modelo final debe quedar en `False`.

In [ ]:
QUICK_TEST = False

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    # En Drive: sobrevive a desconexiones de Colab y evita volver a descargar.
    DATA_DIR = "/content/drive/MyDrive/proyecto_ecg/physionet"
except ImportError:
    DATA_DIR = "physionet"
MODEL_NAME = "cardioalert_cnn_bilstm.keras"

# Tope de ventanas por registro y clase: evita que un paciente con muchas horas domine el dataset.
CAPS = {
    "mitdb": {"normal": 40, "afib": 200},
    "afdb":  {"normal": 80, "afib": 120},
    "edb":   {"normal": 30, "isquemia": 120},
}
EPOCHS = 1 if QUICK_TEST else 40
BATCH = 64
CLASSES = ["normal", "afib", "isquemia"]   # mismo orden que la API y la app

## 3. Preprocesamiento idéntico a la app

Es el port exacto de `CardioAlertMobile/src/signal/preprocess.ts`: remuestreo lineal a 250 Hz, filtro pasa-banda 0.5–40 Hz (pasa-altos de un polo y pasa-bajos ida y vuelta) y z-score por ventana. Se verificó contra el código TypeScript de la app con una diferencia máxima de 6e-8 en 130, 250 y 360 Hz.

In [ ]:
"""Port exacto de CardioAlertMobile/src/signal/preprocess.ts, vectorizado por lotes."""
import numpy as np
from scipy.signal import lfilter

TARGET_SAMPLE_RATE = 250
WINDOW_SECONDS = 10
WINDOW_SIZE = TARGET_SAMPLE_RATE * WINDOW_SECONDS


def resample(x, source_rate, target_rate=TARGET_SAMPLE_RATE):
    """Interpolación lineal igual a `resample` de la app. x: (lote, n)."""
    x = np.atleast_2d(np.asarray(x, dtype=np.float64))
    if source_rate == target_rate:
        return x.copy()
    n = x.shape[1]
    ratio = source_rate / target_rate
    out_len = max(1, int(np.floor(n / ratio + 0.5)))  # Math.round de JS
    pos = np.arange(out_len) * ratio
    left = np.floor(pos).astype(int)
    right = np.minimum(left + 1, n - 1)
    frac = pos - left
    return x[:, left] * (1 - frac) + x[:, right] * frac


def _highpass(x, fs, cutoff):
    # App: y[0] = 0; y[i] = a * (y[i-1] + x[i] - x[i-1])
    rc = 1 / (2 * np.pi * cutoff)
    a = rc / (rc + 1 / fs)
    d = np.diff(x, axis=1, prepend=x[:, :1])  # d[0] = 0
    return lfilter([a], [1, -a], d, axis=1)


def _lowpass(x, fs, cutoff):
    # App: y[0] = x[0]; y[i] = y[i-1] + alpha * (x[i] - y[i-1])
    rc = 1 / (2 * np.pi * cutoff)
    alpha = (1 / fs) / (rc + 1 / fs)
    zi = ((1 - alpha) * x[:, :1])
    y, _ = lfilter([alpha], [1, -(1 - alpha)], x, axis=1, zi=zi)
    return y


def bandpass(x, fs, low=0.5, high=40.0):
    """Pasa-altos y luego pasa-bajos ida y vuelta, igual que la app."""
    hp = _highpass(x, fs, low)
    fwd = _lowpass(hp, fs, high)
    return _lowpass(fwd[:, ::-1], fs, high)[:, ::-1]


def zscore(x):
    mean = x.mean(axis=1, keepdims=True)
    std = x.std(axis=1, keepdims=True)  # poblacional, igual que la app
    return (x - mean) / (std + 1e-8)


def preprocess_windows(raw, source_rate):
    """raw: (lote, 10 s a source_rate) -> (lote, 2500) listo para el modelo."""
    return zscore(bandpass(resample(raw, source_rate), TARGET_SAMPLE_RATE)).astype(np.float32)


# Autoverificación
_t = np.random.default_rng(0).normal(size=(3, 10 * 360))
_p = preprocess_windows(_t, 360)
assert _p.shape == (3, WINDOW_SIZE)
assert np.allclose(_p.mean(axis=1), 0, atol=1e-4) and np.allclose(_p.std(axis=1), 1, atol=1e-3)
print("Preprocesamiento OK:", _p.shape)

## 4. Etiquetado de ventanas

- **MIT-BIH y AFDB**: ventana de 10 s completamente dentro de un ritmo `(N` (normal) o `(AFIB`. En MIT-BIH, las normales además deben tener **solo latidos normales**.
- **European ST-T**: *isquemia* si la ventana está dentro de un episodio de desviación del segmento ST (`ST0±`, `ST1±`), tomada de la derivación donde ocurre el episodio; *normal* si no toca ningún episodio ST ni T y solo tiene latidos normales, de una derivación al azar para que la derivación no delate la clase.
- Las ventanas **no se solapan**: el notebook anterior centraba una ventana en cada latido y generaba casi-copias.

In [ ]:
"""Extracción de ventanas etiquetadas. Cada base aporta Normal + su patología."""
import numpy as np

LABELS = {"normal": 0, "afib": 1, "isquemia": 2}


def clean(note):
    """Las anotaciones de PhysioNet traen un '\x00' al final: '(N\x00' debe leerse '(N'."""
    return note.replace("\x00", "").strip()


def rhythm_intervals(ann, sig_len):
    """Intervalos (inicio, fin, ritmo) a partir de las anotaciones '(N', '(AFIB', ..."""
    marks = [(s, clean(a)) for s, a in zip(ann.sample, ann.aux_note) if clean(a).startswith("(")]
    out = []
    for i, (start, rhythm) in enumerate(marks):
        end = marks[i + 1][0] if i + 1 < len(marks) else sig_len
        out.append((start, end, rhythm[1:]))
    return out


def st_episodes(ann, sig_len):
    """Episodios de la European ST-T: '(ST0+' ... 'ST0+)'. Devuelve {clave: [(ini, fin)]}."""
    open_at, episodes = {}, {}
    for s, a in zip(ann.sample, ann.aux_note):
        a = clean(a)
        if a.startswith("(") and a[1:3] in ("ST", "T0", "T1"):
            open_at[a[1:]] = s
        elif a.endswith(")") and a[:-1] in open_at:
            key = a[:-1]
            episodes.setdefault(key, []).append((open_at.pop(key), s))
    for key, start in open_at.items():  # episodio que sigue abierto al final
        episodes.setdefault(key, []).append((start, sig_len))
    return episodes


def _inside(start, end, intervals):
    return any(a <= start and end <= b for a, b in intervals)


def _overlaps(start, end, intervals):
    return any(start < b and a < end for a, b in intervals)


def label_rhythm_db(record, ann, fs, beats_strict):
    """MIT-BIH y AFDB: ventanas completamente dentro de ritmo normal o AFib, sin solaparse."""
    n = int(WINDOW_SECONDS * fs)
    sig_len = record.sig_len
    cands = {"normal": [], "afib": []}
    beat_ok = None
    if beats_strict:
        beat_idx = np.array(ann.sample)
        beat_sym = np.array(ann.symbol)
    for start, end, rhythm in rhythm_intervals(ann, sig_len):
        label = {"N": "normal", "AFIB": "afib"}.get(rhythm)
        if not label:
            continue
        for s in range(start, end - n + 1, n):
            if beats_strict and label == "normal":
                inside = (beat_idx >= s) & (beat_idx < s + n)
                syms = set(beat_sym[inside]) - {"+", "~", "|", '"'}
                if syms - {"N"}:  # algún latido no normal
                    continue
            cands[label].append(s)
    return cands


def label_edb(record, ann, fs, rng):
    """European ST-T (2 derivaciones).
    Isquemia: ventana dentro de un episodio ST, tomada de la derivación del episodio.
    Normal: sin ningún episodio ST ni T y solo latidos normales, de una derivación al azar,
    para que la derivación no delate la clase. Devuelve {clase: [(inicio, derivación)]}."""
    n = int(WINDOW_SECONDS * fs)
    sig_len = record.sig_len
    eps = st_episodes(ann, sig_len)
    st_by_lead = {lead: eps.get(f"ST{lead}+", []) + eps.get(f"ST{lead}-", []) for lead in (0, 1)}
    any_episode = [iv for ivs in eps.values() for iv in ivs]
    beat_idx = np.array(ann.sample)
    beat_sym = np.array(ann.symbol)
    cands = {"normal": [], "isquemia": []}
    for s in range(0, sig_len - n + 1, n):
        leads = [lead for lead in (0, 1) if _inside(s, s + n, st_by_lead[lead])]
        if leads:
            cands["isquemia"].append((s, leads[0]))
        elif not _overlaps(s, s + n, any_episode):
            inside = (beat_idx >= s) & (beat_idx < s + n)
            syms = set(beat_sym[inside]) - {"+", "~", "|", '"'}
            if not (syms - {"N"}):
                cands["normal"].append((s, int(rng.integers(0, 2))))
    return cands

## 5. Descarga de PhysioNet y extracción

In [ ]:
DBS = {"mitdb": "MIT-BIH Arrhythmia", "afdb": "MIT-BIH Atrial Fibrillation", "edb": "European ST-T"}
QUICK_RECORDS = {"mitdb": ["201"], "afdb": ["04015"], "edb": ["e0103"]}

def records_of(db):
    return QUICK_RECORDS[db] if QUICK_TEST else wfdb.get_record_list(db)

from concurrent.futures import ThreadPoolExecutor

def download(db, rec):
    target = os.path.join(DATA_DIR, db)
    if os.path.exists(os.path.join(target, rec + ".atr")):
        return "cache"
    for attempt in range(3):  # PhysioNet a veces corta la conexión
        try:
            wfdb.dl_database(db, target, records=[rec])
            return "ok"
        except Exception as e:
            last = e
            time.sleep(5 * (attempt + 1))
    return f"error: {last}"

for db in DBS:
    recs = records_of(db)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=8) as pool:  # PhysioNet es lento por archivo: en paralelo rinde mucho más
        results = list(pool.map(lambda r: download(db, r), recs))
    failed = [r for r, res in zip(recs, results) if res.startswith("error")]
    print(f"{db}: {len(recs)} registros ({results.count('cache')} ya estaban) en {time.time() - t0:.0f} s"
          + (f" · fallaron: {failed}" if failed else ""))

In [ ]:
rng = np.random.default_rng(SEED)
X_parts, y_parts, groups, sources = [], [], [], []

for db in DBS:
    for rec in records_of(db):
        path = os.path.join(DATA_DIR, db, rec)
        try:
            ann = wfdb.rdann(path, "atr")
            record = wfdb.rdrecord(path, channels=None if db == "edb" else [0])
        except Exception as e:
            print(f"  omitido {db}/{rec}: {e}")   # afdb tiene registros sin señal
            continue
        fs = record.fs
        sig = np.nan_to_num(record.p_signal.astype(np.float32))
        n = int(WINDOW_SECONDS * fs)

        if db == "edb":
            cands = label_edb(record, ann, fs, rng)
            raw, labels = [], []
            for label, items in cands.items():
                if len(items) > CAPS[db][label]:
                    idx = rng.choice(len(items), CAPS[db][label], replace=False)
                    items = [items[i] for i in idx]
                for s, lead in items:
                    raw.append(sig[s:s + n, lead]); labels.append(LABELS[label])
        else:
            cands = label_rhythm_db(record, ann, fs, beats_strict=(db == "mitdb"))
            raw, labels = [], []
            for label, starts in cands.items():
                starts = np.array(starts, dtype=int)
                if len(starts) > CAPS[db][label]:
                    starts = rng.choice(starts, CAPS[db][label], replace=False)
                for s in starts:
                    raw.append(sig[s:s + n, 0]); labels.append(LABELS[label])

        if not raw:
            continue
        X_parts.append(preprocess_windows(np.stack(raw), fs))
        y_parts.append(np.array(labels))
        groups += [f"{db}/{rec}"] * len(labels)
        sources += [db] * len(labels)

X = np.concatenate(X_parts)[..., None]
y = np.concatenate(y_parts)
groups = np.array(groups); sources = np.array(sources)

print("X:", X.shape, "· pacientes/registros:", len(set(groups)))
for db in DBS:
    m = sources == db
    print(f"  {db:6s}", {c: int(((y == i) & m).sum()) for i, c in enumerate(CLASSES)})
print("Total por clase:", {c: int((y == i).sum()) for i, c in enumerate(CLASSES)})

## 6. División por paciente

`StratifiedGroupKFold` agrupa por registro (paciente): todas las ventanas de un paciente quedan en un solo conjunto. Se reserva ~20% de pacientes para prueba y, del resto, ~20% para validación.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def group_split(y, groups, n_splits, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return next(sgkf.split(np.zeros(len(y)), y, groups))

n_splits = 2 if QUICK_TEST else 5
trval_idx, test_idx = group_split(y, groups, n_splits, SEED)
tr_rel, va_rel = group_split(y[trval_idx], groups[trval_idx], n_splits, SEED + 1)
train_idx, val_idx = trval_idx[tr_rel], trval_idx[va_rel]

# Ningún paciente puede aparecer en dos conjuntos
assert not set(groups[train_idx]) & set(groups[test_idx])
assert not set(groups[val_idx]) & set(groups[test_idx])
assert not set(groups[train_idx]) & set(groups[val_idx])

for name, idx in [("TRAIN", train_idx), ("VAL", val_idx), ("TEST", test_idx)]:
    print(f"{name:5s} pacientes={len(set(groups[idx])):3d} ventanas={len(idx):6d}",
          {c: int((y[idx] == i).sum()) for i, c in enumerate(CLASSES)})

## 7. Aumento de datos

Solo en entrenamiento: inversión de polaridad (la banda Polar puede quedar en otra orientación que el Holter) y ruido gaussiano leve (movimiento en la ambulancia).

In [ ]:
def augment(x, label):
    flip = tf.where(tf.random.uniform([]) < 0.5, -1.0, 1.0)
    x = x * flip + tf.random.normal(tf.shape(x), stddev=0.05)
    return x, label

train_ds = (tf.data.Dataset.from_tensor_slices((X[train_idx], y[train_idx]))
            .shuffle(len(train_idx), seed=SEED).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH).prefetch(tf.data.AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((X[val_idx], y[val_idx])).batch(BATCH)

## 8. Modelo CNN-BiLSTM

La CNN extrae la **forma** de cada latido (QRS, segmento ST); la BiLSTM analiza la **secuencia** de latidos en ambos sentidos (irregularidad del ritmo en la AFib).

In [ ]:
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.utils.class_weight import compute_class_weight

def conv_block(x, filters, kernel):
    x = layers.Conv1D(filters, kernel, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return layers.MaxPooling1D(2)(x)

inputs = layers.Input(shape=(WINDOW_SIZE, 1))
x = conv_block(inputs, 32, 7)
x = conv_block(x, 64, 5)
x = conv_block(x, 128, 5)
x = conv_block(x, 128, 3)                      # 2500 -> 156 pasos de tiempo
x = layers.Bidirectional(layers.LSTM(64))(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
model = models.Model(inputs, outputs, name="cardioalert_cnn_bilstm")

model.compile(optimizer=optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

weights = compute_class_weight("balanced", classes=np.unique(y[train_idx]), y=y[train_idx])
class_weight = {int(c): float(w) for c, w in zip(np.unique(y[train_idx]), weights)}
print("Pesos de clase:", class_weight)

In [ ]:
history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    ],
    verbose=2,
)

## 9. Evaluación con pacientes nunca vistos

Estas son las cifras que van a la tesis. Metas del proyecto: precisión global > 85 %, sensibilidad para AFib > 80 % y para isquemia > 80 %.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score
import matplotlib.pyplot as plt

probs = model.predict(X[test_idx], batch_size=256, verbose=0)
pred = probs.argmax(axis=1); true = y[test_idx]
labels_present = [i for i in range(len(CLASSES)) if i in set(true)]

print(classification_report(true, pred, labels=labels_present,
                            target_names=[CLASSES[i] for i in labels_present], digits=3, zero_division=0))

acc = accuracy_score(true, pred)
sens = recall_score(true, pred, labels=labels_present, average=None, zero_division=0)
print(f"Precisión global: {acc:.1%}  (meta > 85%)")
for i, s in zip(labels_present, sens):
    meta = "" if CLASSES[i] == "normal" else "  (meta > 80%)"
    print(f"Sensibilidad {CLASSES[i]:9s}: {s:.1%}{meta}")

cm = confusion_matrix(true, pred, labels=labels_present)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(labels_present)), [CLASSES[i] for i in labels_present])
ax.set_yticks(range(len(labels_present)), [CLASSES[i] for i in labels_present])
ax.set_xlabel("Predicción"); ax.set_ylabel("Real"); ax.set_title("Matriz de confusión (test)")
for (r, c), v in np.ndenumerate(cm):
    ax.text(c, r, v, ha="center", va="center")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, key in zip(axes, ["accuracy", "loss"]):
    ax.plot(history.history[key], label="entrenamiento"); ax.plot(history.history["val_" + key], label="validación")
    ax.set_title(key); ax.set_xlabel("época"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Control de atajos: ¿el modelo reconoce la base de datos?

Si el modelo hubiera aprendido la base en lugar de la patología, los **normales** de una base se clasificarían muy distinto a los de otra. Con el dataset corregido, la exactitud en normales debería ser parecida entre bases.

In [ ]:
src_test = sources[test_idx]
print(f"{'base':8s}{'ventanas':>10s}{'exactitud':>11s}{'normales bien':>16s}")
for db in DBS:
    m = src_test == db
    if not m.any():
        continue
    normal = m & (true == 0)
    ok_normal = f"{(pred[normal] == 0).mean():.1%}" if normal.any() else "—"
    print(f"{db:8s}{m.sum():>10d}{(pred[m] == true[m]).mean():>11.1%}{ok_normal:>16s}")

### Prueba de cordura: corazón sano sintético

El modelo anterior clasificaba un ECG sano como AFib o isquemia al 100%. Aquí se genera el **mismo** ritmo sinusal regular (con onda P, 75 BPM) a las tres frecuencias originales y se pasa por el preprocesamiento de la app. Un modelo que aprendió la patología debería inclinarse por *normal* en los tres casos. Las señales sintéticas no son pacientes reales: esto es una alarma temprana, no una validación.

In [ ]:
def synthetic_healthy(fs, bpm=75, seconds=WINDOW_SECONDS):
    t = np.arange(int(seconds * fs)) / fs
    phase = t % (60 / bpm)
    g = lambda c, w, a: a * np.exp(-((phase - c) ** 2) / (2 * w * w))
    x = g(0.10, 0.025, 0.15) + g(0.20, 0.008, -0.12) + g(0.22, 0.010, 1.3) + g(0.245, 0.010, -0.25) + g(0.42, 0.045, 0.32)
    return x + np.random.default_rng(0).normal(0, 0.02, x.size)

for fs in (130, 250, 360):
    p = model.predict(preprocess_windows(synthetic_healthy(fs)[None], fs)[..., None], verbose=0)[0]
    print(f"{fs} Hz -> {CLASSES[p.argmax()]:9s}", {c: f"{v:.0%}" for c, v in zip(CLASSES, p)})

## 10. Exportar el modelo

Guarda `cardioalert_cnn_bilstm.keras`. En Colab también lo copia a Drive (`MyDrive/proyecto_ecg/`) y lo descarga. Ese archivo va en `ml-api/model/` del repositorio.

In [ ]:
model.save(MODEL_NAME)
size_kb = os.path.getsize(MODEL_NAME) / 1024
print(f"Guardado {MODEL_NAME} ({size_kb:.0f} KB)")

# Verificación: el archivo recarga y predice igual
reloaded = tf.keras.models.load_model(MODEL_NAME)
assert np.allclose(reloaded.predict(X[test_idx[:8]], verbose=0), model.predict(X[test_idx[:8]], verbose=0), atol=1e-5)
print("Recarga verificada · entrada", reloaded.input_shape, "· salida", reloaded.output_shape)

summary = {
    "modelo": MODEL_NAME, "tensorflow": tf.__version__, "ventanas_total": int(len(y)),
    "pacientes_test": int(len(set(groups[test_idx]))), "precision_test": round(float(acc), 4),
    "sensibilidad": {CLASSES[i]: round(float(s), 4) for i, s in zip(labels_present, sens)},
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/proyecto_ecg"
    os.makedirs(dest, exist_ok=True)
    tf.io.gfile.copy(MODEL_NAME, f"{dest}/{MODEL_NAME}", overwrite=True)
    json.dump(summary, open(f"{dest}/metricas_{MODEL_NAME}.json", "w"), indent=2, ensure_ascii=False)
    print("Copiado a Drive:", dest)
    files.download(MODEL_NAME)
except ImportError:
    print("Fuera de Colab: el archivo quedó en", os.path.abspath(MODEL_NAME))